# Patient Records — Data Quality & Cleaning

Load `patient_records_raw.csv` and work through the checklist: structural checks, disguised missing values, duplicates, categorical/numeric validation, and cross-column consistency.

**Source:** `../backend/app/data/patient_records_raw.csv`

## Setup (run this first)

If imports fail, your notebook is using the wrong Python. In Cursor:

1. Click the kernel name in the **top-right** of the notebook (e.g. `Python 3.x`)
2. Choose **Select Another Kernel...**
3. Pick **Python Environments...**
4. Select **`.venv (Python 3.12)`** under `missing-data-pipeline/backend`

Then run the cell below — it should print a path ending in `backend\.venv\Scripts\python.exe`.

In [64]:
import sys

print("Python:", sys.executable)
print("Version:", sys.version)

try:
    import numpy as np
    import pandas as pd
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("OK — correct environment")
except ModuleNotFoundError as e:
    print("ERROR:", e)
    print("Switch kernel to missing-data-pipeline/backend/.venv (see Setup above)")

Python: c:\Users\amalj\OneDrive\Desktop\dissertation\missing-data-pipeline\backend\.venv\Scripts\python.exe
Version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
numpy: 2.5.1
pandas: 3.0.3
OK — correct environment


In [65]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("../backend/app/data/patient_records_raw.csv")
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Loaded {len(df):,} rows from {DATA_PATH.resolve()}")
df.head()

Loaded 429 rows from C:\Users\amalj\OneDrive\Desktop\dissertation\missing-data-pipeline\backend\app\data\patient_records_raw.csv


,patient_id,age,gender,bmi,systolic_bp,glucose,smoking_status,region,visits_last_year,severity_score
0,P1375,19,Male,35.8,0.0,77.6,Former,West,1,3.9
1,P1268,23,Female,21.7,121.0,115.6,Never,South,7,6.8
2,P1228,38,Male,19.9,128.0,96.8,Current,North,2,4.2
3,P1162,68,Female,30.9,131.0,98.0,Never,East,0,2.0
4,P1180,65,Male,24.4,131.0,83.1,Never,South,2,6.4


## 1. Structural checks (do these first — before touching any values)

- Check shape (`df.shape`) — does row/column count look sane?
- Check dtypes (`df.info()`) — is every column the type you expect?
- Check column names — consistent casing, no stray whitespace, no duplicate column names

In [66]:
print("Shape:", df.shape)
print("\nColumn names:", list(df.columns))
print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())
print("Columns with leading/trailing whitespace:", [c for c in df.columns if c != c.strip()])
df.info()

Shape: (429, 10)

Column names: ['patient_id', 'age', 'gender', 'bmi', 'systolic_bp', 'glucose', 'smoking_status', 'region', 'visits_last_year', 'severity_score']
Duplicate column names: []
Columns with leading/trailing whitespace: []
<class 'pandas.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   patient_id        429 non-null    str    
 1   age               429 non-null    int64  
 2   gender            429 non-null    str    
 3   bmi               424 non-null    float64
 4   systolic_bp       419 non-null    float64
 5   glucose           420 non-null    float64
 6   smoking_status    420 non-null    str    
 7   region            429 non-null    str    
 8   visits_last_year  429 non-null    int64  
 9   severity_score    429 non-null    float64
dtypes: float64(4), int64(2), str(4)
memory usage: 33.6 KB


## 2. Find hidden/disguised missing values

Scan for string placeholders and numeric sentinels, then convert them to real `NaN` so pandas can track them.

> **Note:** This catches repeated placeholder values (e.g. `systolic_bp = 0`). One-off implausible values injected in the data (e.g. `age = -3`, `bmi = 187.4`) are **not** sentinels — section 5 flags and drops those instead.

In [67]:
MISSING_STRINGS = {"", "NA", "N/A", "null", "none", "-", "?", "unknown", "missing"}
NUMERIC_SENTINELS = {0: ["systolic_bp"], -1: [], 999: [], 9999: []}

object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    normalized = df[col].astype(str).str.strip().str.lower()
    mask = normalized.isin({s.lower() for s in MISSING_STRINGS})
    if mask.any():
        print(f"{col}: string placeholders -> {mask.sum()}")
        df.loc[mask, col] = np.nan

for sentinel, cols in NUMERIC_SENTINELS.items():
    for col in cols:
        if col in df.columns:
            mask = df[col] == sentinel
            if mask.any():
                print(f"{col}: sentinel {sentinel} -> {mask.sum()}")
                df.loc[mask, col] = np.nan

print("\nDisguised missing values converted to NaN.")

systolic_bp: sentinel 0 -> 3

Disguised missing values converted to NaN.


C:\Users\amalj\AppData\Local\Temp\ipykernel_11852\2763702875.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


## 3. Duplicates

- Fully duplicate rows
- Duplicates ignoring the ID column
- Duplicate IDs with different data

In [68]:
id_col = "patient_id"

full_dupes = df.duplicated().sum()
dupes_without_id = df.drop(columns=[id_col]).duplicated().sum()
duplicate_ids = df[id_col].duplicated().sum()
conflicting_ids = df[id_col].duplicated(keep=False)

print(f"Fully duplicate rows: {full_dupes}")
print(f"Duplicate rows (ignoring {id_col}): {dupes_without_id}")
print(f"Duplicate {id_col} values: {duplicate_ids}")

if conflicting_ids.any():
    display(df[conflicting_ids].sort_values(id_col))
else:
    print("No duplicate IDs with conflicting rows.")

Fully duplicate rows: 7
Duplicate rows (ignoring patient_id): 8
Duplicate patient_id values: 8


,patient_id,age,gender,bmi,systolic_bp,glucose,smoking_status,region,visits_last_year,severity_score
291,P1015,73,Male,18.0,115.0,72.1,Former,East,4,5.5
330,P1015,73,Male,18.0,115.0,72.1,Former,East,4,5.5
285,P1073,42,Male,27.5,121.0,96.8,Former,West,3,5.1
315,P1073,42,Male,27.5,121.0,96.8,Former,West,3,5.1
185,P1116,56,Female,25.3,125.0,79.5,Current,South,5,5.4
190,P1116,56,Female,25.3,125.0,79.5,Current,South,5,5.4
28,P1124,54,Female,25.9,124.0,112.6,Never,West,3,4.2
248,P1124,54,Female,25.9,124.0,112.6,Never,West,3,4.2
215,P1127,28,Female,20.1,118.0,NaN,Former,East,3,4.6
246,P1127,28,Female,20.1,118.0,54.5,Former,East,3,4.6


### Action — drop duplicate rows

Review the counts above, then remove duplicates (keeping the first occurrence).

In [69]:
before = len(df)
df = df.drop_duplicates(subset=df.columns.difference([id_col]), keep="first")
print(f"Dropped {before - len(df)} duplicate rows")
print("Shape after deduplication:", df.shape)

Dropped 8 duplicate rows
Shape after deduplication: (421, 10)


## 4. Categorical consistency

Inspect unique values, standardise casing/spelling, and flag invalid categories.

In [70]:
categorical_cols = ["gender", "smoking_status", "region"]

for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col} unique values:")
        print(df[col].value_counts(dropna=False))

GENDER_MAP = {
    "male": "Male", "m": "Male",
    "female": "Female", "f": "Female",
}
SMOKING_MAP = {
    "current": "Current", "curent": "Current", "currnt": "Current",
    "never": "Never", "neverr": "Never", "nver": "Never",
    "former": "Former", "formerr": "Former", "fromer": "Former",
}

if "gender" in df.columns:
    df["gender"] = (
        df["gender"].astype(str).str.strip().str.lower()
        .replace({"nan": np.nan}).replace(GENDER_MAP)
    )

if "smoking_status" in df.columns:
    df["smoking_status"] = (
        df["smoking_status"].astype(str).str.strip().str.lower()
        .replace({"nan": np.nan}).replace(SMOKING_MAP)
    )

if "region" in df.columns:
    df["region"] = (
        df["region"].astype(str).str.strip().str.title()
        .replace({"Nan": np.nan})
    )

print("\nAfter standardisation:")
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col}:", sorted(df[col].dropna().unique()))


gender unique values:
gender
Male       202
Female     194
F            6
female       5
FEMALE       4
male         4
 Male        3
 Female      1
MALE         1
M            1
Name: count, dtype: int64

smoking_status unique values:
smoking_status
Never      213
Former      99
Current     83
NaN          9
never        5
current      2
NEVER        2
Nver         2
Curent       1
FORMER       1
CURRENT      1
currnt       1
Neverr       1
former       1
Name: count, dtype: int64

region unique values:
region
West      107
East      103
South      99
North      97
north       3
south       2
EAST        2
 West       2
NORTH       2
 South      1
SOUTH       1
east        1
west        1
Name: count, dtype: int64

After standardisation:

gender: ['Female', 'Male']

smoking_status: ['Current', 'Former', 'Never']

region: ['East', 'North', 'South', 'West']


## 5. Numeric plausibility

Check min/max ranges and flag implausible values.

In [71]:
numeric_cols = ["age", "bmi", "systolic_bp", "glucose", "visits_last_year", "severity_score"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

display(df[numeric_cols].describe())

plausibility_rules = {
    "age": (0, 120),
    "bmi": (10, 80),
    "systolic_bp": (50, 250),
    "glucose": (20, 500),
    "visits_last_year": (0, 50),
    "severity_score": (0, 10),
}

for col, (low, high) in plausibility_rules.items():
    if col not in df.columns:
        continue
    mask = df[col].notna() & ((df[col] < low) | (df[col] > high))
    if mask.any():
        print(f"{col}: {mask.sum()} values outside [{low}, {high}]")
        display(df.loc[mask, [id_col, col]])

,age,bmi,systolic_bp,glucose,visits_last_year,severity_score
count,421.000000,416.000000,408.000000,412.000000,421.000000,421.000000
mean,52.040380,27.050962,125.031863,100.201214,2.978622,4.980285
std,19.948046,12.289890,14.564656,26.567238,1.754455,1.933173
min,-3.000000,9.100000,90.000000,30.500000,0.000000,0.000000
25%,34.000000,22.775000,115.000000,81.600000,2.000000,3.700000
50%,53.000000,26.600000,124.000000,99.350000,3.000000,5.100000
75%,70.000000,29.800000,135.000000,118.725000,4.000000,6.300000
max,84.000000,187.400000,164.000000,170.700000,9.000000,10.000000


age: 1 values outside [0, 120]


,patient_id,age
34,P1057,-3


bmi: 3 values outside [10, 80]


,patient_id,bmi
140,P1130,187.4
233,P1288,187.4
359,P1043,9.1


### Action — drop implausible values

Review the flagged rows above, then remove values outside the plausibility ranges.

In [72]:
before = len(df)
for col, (low, high) in plausibility_rules.items():
    if col in df.columns:
        mask = df[col].notna() & ((df[col] < low) | (df[col] > high))
        df = df[~mask]
print(f"Dropped {before - len(df)} rows with implausible values")
print("Shape after dropping implausible values:", df.shape)

Dropped 4 rows with implausible values
Shape after dropping implausible values: (417, 10)


## 6. Assess real missing values

Now that disguised missing values are converted, count true `NaN`s per column and decide on a strategy.

In [73]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
display(missing_summary[missing_summary["missing_count"] > 0])

rows_with_any_missing = df.isna().any(axis=1).sum()
print(f"Rows with at least one missing value: {rows_with_any_missing} ({rows_with_any_missing / len(df) * 100:.1f}%)")

,missing_count,missing_pct
systolic_bp,13,3.12
glucose,9,2.16
smoking_status,9,2.16
bmi,5,1.20


Rows with at least one missing value: 35 (8.4%)


### Action — drop rows with missing values

Review the missing-value summary above, then drop rows to protect ground truth.

In [74]:
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows with missing values")
print("Shape after dropping missing values:", df.shape)

Dropped 35 rows with missing values
Shape after dropping missing values: (382, 10)


## 7. Cross-column logical consistency

Check for logically impossible combinations across columns.

In [75]:
expected_regions = {"North", "South", "East", "West"}
unexpected_regions = set(df["region"].dropna().unique()) - expected_regions
print("Unexpected region values:", unexpected_regions or "none")

expected_genders = {"Male", "Female"}
unexpected_genders = set(df["gender"].dropna().unique()) - expected_genders
print("Unexpected gender values:", unexpected_genders or "none")

expected_smoking = {"Never", "Current", "Former"}
unexpected_smoking = set(df["smoking_status"].dropna().unique()) - expected_smoking
print("Unexpected smoking_status values:", unexpected_smoking or "none")

Unexpected region values: none
Unexpected gender values: none
Unexpected smoking_status values: none


## 8. Final sanity pass

Re-run summary stats on the cleaned result and record how much data was retained.

In [76]:
print("Original shape:", df_raw.shape)
print("Current shape:", df.shape)
print(f"Rows retained: {len(df) / len(df_raw) * 100:.1f}%")

df.info()
display(df.describe(include="all").T)

Original shape: (429, 10)
Current shape: (382, 10)
Rows retained: 89.0%
<class 'pandas.DataFrame'>
Index: 382 entries, 1 to 428
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   patient_id        382 non-null    str    
 1   age               382 non-null    int64  
 2   gender            382 non-null    str    
 3   bmi               382 non-null    float64
 4   systolic_bp       382 non-null    float64
 5   glucose           382 non-null    float64
 6   smoking_status    382 non-null    str    
 7   region            382 non-null    str    
 8   visits_last_year  382 non-null    int64  
 9   severity_score    382 non-null    float64
dtypes: float64(4), int64(2), str(4)
memory usage: 32.8 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
patient_id,382,382,P1268,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,382.0,NaN,NaN,NaN,52.332461,19.674836,18.0,34.0,52.5,70.0,84.0
gender,382,2,Female,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bmi,382.0,NaN,NaN,NaN,26.40445,5.112992,11.8,22.825,26.7,29.8,41.0
systolic_bp,382.0,NaN,NaN,NaN,125.062827,14.765933,90.0,115.0,124.0,135.0,164.0
glucose,382.0,NaN,NaN,NaN,100.249476,26.188869,30.5,82.1,99.5,118.7,170.7
smoking_status,382,3,Never,206,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,382,4,West,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
visits_last_year,382.0,NaN,NaN,NaN,2.997382,1.757621,0.0,2.0,3.0,4.0,9.0
severity_score,382.0,NaN,NaN,NaN,5.015969,1.934459,0.0,3.8,5.1,6.375,10.0


## 9. Save cleaned dataset

In [77]:
OUT_PATH = Path("../backend/app/data/patient_records_clean.csv")
df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUT_PATH.resolve()}")
print(f"Final shape: {df.shape} ({len(df) / len(df_raw) * 100:.1f}% of original rows retained)")

Saved cleaned dataset to C:\Users\amalj\OneDrive\Desktop\dissertation\missing-data-pipeline\backend\app\data\patient_records_clean.csv
Final shape: (382, 10) (89.0% of original rows retained)
